In [1]:
import os
print(os.getcwd())

C:\Users\00873\M&Q


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

In [5]:
try:
    df = pd.read_excel("C:\\Users\\00873\\M&Q\\Tablas a Migrar V1.xlsx")
    print(df.shape)
except FileNotFoundError:
    print("No se encontró el archivo. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error: {e}")


(43, 8)


In [7]:
df.head(13)

,SOLUCIÓN A LA QUE PERTENECE / ARQUITECTURA,SISTEMA DE INFORMACIÓN,OWNER,BASEDATOS,CANTIDADTABLAS,CANTREGISTROS,CANTCOLUMNAS,TAMAÑOGB
0,MUISCA,ANOTACIONES FISCALES,ANOFIS,SUMMA,2,3.000000e+00,100.0,0.000000
1,MUISCA,PROFESIONALES DE CAMBIO,CAMBIARIO,SUMMA,93,2.244360e+05,9358.0,0.069012
2,MUISCA,CARTERA,CARTERA,SIGMA,27,2.530364e+07,1971.0,1.122722
3,MUISCA,DEVOLUCIONES Y COMPENSACIONES,CDEVOL,SIGMA,315,2.773299e+09,12134.0,816.417165
4,MUISCA,CONTROL ENTIDADES RECAUDADORAS,CTLEAR,SIGMA,13,9.186446e+07,561.0,13.740361
5,MUISCA,DENUNCIAS FISCALIZACIÓN,DENFIS,SUMMA,82,1.129502e+06,4732.0,0.494723
6,MUISCA,FISCALIZACIÓN VIAJEROS,FISVIA,PRIMA,1,2.932572e+07,488.0,29.551270
7,MUISCA,LABORATORIO DE ADUANAS,LABORATORIO,SUMMA,77,1.472524e+06,1885.0,0.256033
8,MUISCA,SALIDA DE MERCANCÍAS - EXPORTACIONES,MADU,PRIMA,349,7.832825e+09,20646.0,2479.254457
9,MUISCA,ARANCEL,MARA,PRIMA,221,2.260673e+08,21227.0,41.389503


In [5]:
df_sistemas = pd.read_excel('Tablas a Migrar V1 - copia.xlsx', sheet_name='Sistemas a Migrar')
df_detalle = pd.read_excel('Tablas a Migrar V1 - copia.xlsx', sheet_name='DetalleTablas')


In [8]:
df_sistemas.head(1)

,SOLUCIÓN A LA QUE PERTENECE / ARQUITECTURA,SISTEMA DE INFORMACIÓN,OWNER,BASEDATOS,CANTIDADTABLAS,CANTREGISTROS,CANTCOLUMNAS,TAMAÑOGB
0,MUISCA,ANOTACIONES FISCALES,ANOFIS,SUMMA,2,3.0,100.0,0.0


In [74]:
df_detalle.head(10)

,OWNER,BASEDATOS,NOMBRE_TABLA,CANT_TABLAS,CANT_COLUMNAS,REGISTROS,AVG_ROW_LEN,SIZE_GB,TIEMPO_HORAS_1_DEV
0,ANOFIS,SUMMA,ARQ_SERVICIOS_SEGMENTO,1,50,2,71,0.000000,0.000000
1,ANOFIS,SUMMA,ARQ_NUMERADORES,1,50,1,70,0.000000,0.000000
2,CAMBIARIO,SUMMA,EYS_MARCAS_DOC_ES,1,16,25049,94,0.002193,0.001097
3,CAMBIARIO,SUMMA,EYS_MARCAS_DOC_ES,1,48,22769,94,0.001993,0.000996
4,CAMBIARIO,SUMMA,EYS_LOG_ESTADOS_PROC_DOC,1,116,13199,153,0.001881,0.000941
5,CAMBIARIO,SUMMA,EYS_DOCUMENTOS_ES,1,261,10597,199,0.001964,0.000982
6,CAMBIARIO,SUMMA,EYS_DOCUMENTOS_ES1,1,44,10546,2712,0.026637,0.013319
7,CAMBIARIO,SUMMA,EYS_DOCUMENTOS_ES,1,87,10258,199,0.001901,0.000950
8,CAMBIARIO,SUMMA,EYS_AUDITORIA_REGISTROS,1,84,7838,439,0.003205,0.001602
9,CAMBIARIO,SUMMA,GEX_ESTADOS_TRAMITES_EJE,1,12,7315,44,0.000300,0.000150


In [9]:
total_tablas_sistemas = df_sistemas['CANTIDADTABLAS'].sum()
print(f"Total de tablas según SUM(E2:E43): {total_tablas_sistemas}")

Total de tablas según SUM(E2:E43): 10090


In [10]:
# 2. Calcular tamaño total de la migración
tamaño_total_gb = df_sistemas['TAMAÑOGB'].sum()
print(f"Tamaño total estimado (GB): {tamaño_total_gb:.2f}\n")

Tamaño total estimado (GB): 28397.72



In [11]:
resumen_sistemas = df_sistemas.groupby('SISTEMA DE INFORMACIÓN').agg(
    Total_Tablas=('CANTIDADTABLAS', 'sum'),
    Tamaño_GB=('TAMAÑOGB', 'sum')
).reset_index()

In [12]:
print("--- Resumen por Sistema ---")
print(resumen_sistemas.to_markdown(index=False))

--- Resumen por Sistema ---
| SISTEMA DE INFORMACIÓN                                                                 |   Total_Tablas |   Tamaño_GB |
|:---------------------------------------------------------------------------------------|---------------:|------------:|
| ADMINISTRACIÓN                                                                         |            233 |    0        |
| ANOTACIONES FISCALES                                                                   |              2 |    0        |
| ARANCEL                                                                                |            221 |   41.3895   |
| CAPACIDAD OPERATIVA                                                                    |            226 |   35.9188   |
| CARGA IMPORTACIONES                                                                    |            317 | 4374.74     |
| CARTERA                                                                                |             27 |    1.12272

In [13]:
# 4. Resumen por base de datos (DetalleTablas)
resumen_bd = df_detalle.groupby(['OWNER', 'BASEDATOS']).agg(
    Total_Tablas=('NOMBRE_TABLA', 'nunique'),
    Total_Registros=('REGISTROS', 'sum'),
    Tamaño_GB=('SIZE_GB', 'sum')
).reset_index()

In [14]:
print("\n--- Resumen por Base de Datos ---")
print(resumen_bd.to_markdown(index=False))


--- Resumen por Base de Datos ---
| OWNER       | BASEDATOS   |   Total_Tablas |   Total_Registros |   Tamaño_GB |
|:------------|:------------|---------------:|------------------:|------------:|
| ANOFIS      | SUMMA       |              2 |                 3 |    0        |
| CAMBIARIO   | SUMMA       |             60 |            224436 |    0.069012 |
| CARTERA     | SIGMA       |             27 |          25303640 |    1.12272  |
| CDEVOL      | SIGMA       |            227 |        2773298623 |  816.417    |
| CTLEAR      | SIGMA       |             13 |          91864458 |   13.7404   |
| DENFIS      | SUMMA       |             31 |           1129502 |    0.494723 |
| FISVIA      | PRIMA       |              1 |          29325725 |   29.5513   |
| LABORATORIO | SUMMA       |             30 |           1472524 |    0.256033 |
| MADU        | PRIMA       |            240 |        7832824745 | 2479.25     |
| MARA        | PRIMA       |            197 |         226067290 |   41.38

In [15]:
sistemas_validos = df_sistemas['SISTEMA DE INFORMACIÓN'].unique()
tablas_fuera_sistema = df_detalle[~df_detalle['OWNER'].isin(sistemas_validos)]

In [16]:
if not tablas_fuera_sistema.empty:
    print("\n ⚠️ Alertas: Tablas con OWNER no listado en Sistemas a Migrar:")
    print(tablas_fuera_sistema[['OWNER', 'NOMBRE_TABLA']].to_markdown(index=False))
else:
    print("\n✅ Todas las tablas están asociadas a sistemas válidos.")


 ⚠️ Alertas: Tablas con OWNER no listado en Sistemas a Migrar:
| OWNER       | NOMBRE_TABLA                                |
|:------------|:--------------------------------------------|
| ANOFIS      | ARQ_SERVICIOS_SEGMENTO                      |
| ANOFIS      | ARQ_NUMERADORES                             |
| CAMBIARIO   | EYS_MARCAS_DOC_ES                           |
| CAMBIARIO   | EYS_MARCAS_DOC_ES                           |
| CAMBIARIO   | EYS_LOG_ESTADOS_PROC_DOC                    |
| CAMBIARIO   | EYS_DOCUMENTOS_ES                           |
| CAMBIARIO   | EYS_DOCUMENTOS_ES1                          |
| CAMBIARIO   | EYS_DOCUMENTOS_ES                           |
| CAMBIARIO   | EYS_AUDITORIA_REGISTROS                     |
| CAMBIARIO   | GEX_ESTADOS_TRAMITES_EJE                    |
| CAMBIARIO   | EYS_LOG_ESTADOS_PROC_DOC_DEF                |
| CAMBIARIO   | GEX_PARAM_EXPEDIENTES                       |
| CAMBIARIO   | GEX_PARAM_EVENTOS                           |
| CAMB

### VELOCIDAD DE MIGRACION

In [76]:
# --------------------------------------------------------------------------------
# 1. PARÁMETROS DE LA MIGRACIÓN (ajustar según la realidad de su proyecto)
# --------------------------------------------------------------------------------
# Velocidad de migración promedio (GB/hora) suponiendo un desarrollador.
velocidad_migracion_gb_hora = 2.0  

# Horas de trabajo diarias por desarrollador
horas_por_dia = 8  

# Días límite para completar la migración (si se desea calcular el número de desarrolladores requeridos)
dias_limite = 180
# Dias laborables al mes = 20
# 
# 9 meses
# Si se fija un número de desarrolladores, se puede calcular el tiempo resultante.

#   a) Calcular cuántos desarrolladores se necesitan para cumplir 'dias_limite'
#   b) Calcular cuántos días se necesitan si se cuenta con X desarrolladores
desarrolladores_fijos = 9  # Ej: 5 si desea forzar 5 devs y calcular días, o dejar None para calcular devs


In [76]:
# --------------------------------------------------------------------------------
# 1. PARÁMETROS DE LA MIGRACIÓN (ajustar según la realidad de su proyecto)
# --------------------------------------------------------------------------------
# Velocidad de migración promedio (GB/hora) suponiendo un desarrollador.
velocidad_migracion_gb_hora = 2.0  

# Horas de trabajo diarias por desarrollador
horas_por_dia = 8  

# Días límite para completar la migración (si se desea calcular el número de desarrolladores requeridos)
dias_limite = 180
# Dias laborables al mes = 20
# 
# 9 meses
# Si se fija un número de desarrolladores, se puede calcular el tiempo resultante.
# En este ejemplo, solo se usará uno de los dos modos:
#   a) Calcular cuántos desarrolladores se necesitan para cumplir 'dias_limite'
#   b) Calcular cuántos días se necesitan si se cuenta con X desarrolladores
desarrolladores_fijos = 9  # Ej: 5 si desea forzar 5 devs y calcular días, o dejar None para calcular devs


In [78]:
df_detalle['TIEMPO_HORAS_1_DEV'] = df_detalle['SIZE_GB'] / velocidad_migracion_gb_hora

# Sumamos el tiempo total (si lo hiciera 1 solo desarrollador, en secuencia)
tiempo_total_secuencial_horas = df_detalle['TIEMPO_HORAS_1_DEV'].sum()


In [70]:
if desarrolladores_fijos is None:
    # Número de horas totales de trabajo disponibles si se desea terminar en 'dias_limite'
    # con un número desconocido de desarrolladores:
    horas_totales_disponibles = dias_limite * horas_por_dia
    
    # Si se quiere trabajar en paralelo y terminar en 'dias_limite', 
    # se necesita: (tiempo_total_secuencial_horas / horas_totales_disponibles) desarrolladores.
    desarrolladores_requeridos = np.ceil(tiempo_total_secuencial_horas / horas_totales_disponibles)
    
    print(f"\n--- Cálculo de desarrolladores necesarios para completar en {dias_limite} días ---")
    print(f"Tiempo total migración (secuencial, 1 dev): {tiempo_total_secuencial_horas:.2f} horas")
    print(f"Desarrolladores requeridos (aprox): {int(desarrolladores_requeridos)}\n")
    
    # Guardamos en una variable para usar en visualizaciones
    num_devs = desarrolladores_requeridos

else:
    num_devs = desarrolladores_fijos
    # Si tenemos 'desarrolladores_fijos', el trabajo se reparte paralelamente.
    # Modelo simplificado: El tiempo total se divide entre la cantidad de desarrolladores (sin considerar solapamientos u orden).
    tiempo_total_parallel = tiempo_total_secuencial_horas / num_devs
    
    # Convertimos a días asumiendo horas_por_dia
    dias_requeridos = tiempo_total_parallel / horas_por_dia

    print(f"\n--- Cálculo de días necesarios con {num_devs} desarrollador(es) ---")
    print(f"Tiempo total migración (secuencial, 1 dev): {tiempo_total_secuencial_horas:.2f} horas")
    print(f"Tiempo migración en paralelo con {num_devs} devs: {tiempo_total_parallel:.2f} horas (~ {dias_requeridos:.1f} días)\n")


--- Cálculo de días necesarios con 3 desarrollador(es) ---
Tiempo total migración (secuencial, 1 dev): 14198.86 horas
Tiempo migración en paralelo con 3 devs: 4732.95 horas (~ 591.6 días)



In [80]:
if desarrolladores_fijos is None:
    # Número de horas totales de trabajo disponibles si se desea terminar en 'dias_limite'
    # con un número desconocido de desarrolladores:
    horas_totales_disponibles = dias_limite * horas_por_dia
    
    # Si se quiere trabajar en paralelo y terminar en 'dias_limite', 
    # se necesita: (tiempo_total_secuencial_horas / horas_totales_disponibles) desarrolladores.
    desarrolladores_requeridos = np.ceil(tiempo_total_secuencial_horas / horas_totales_disponibles)
    
    print(f"\n--- Cálculo de desarrolladores necesarios para completar en {dias_limite} días ---")
    print(f"Tiempo total migración (secuencial, 1 dev): {tiempo_total_secuencial_horas:.2f} horas")
    print(f"Desarrolladores requeridos (aprox): {int(desarrolladores_requeridos)}\n")
    
    # Guardamos en una variable para usar en visualizaciones
    num_devs = desarrolladores_requeridos

else:
    num_devs = desarrolladores_fijos
    # Si tenemos 'desarrolladores_fijos', el trabajo se reparte paralelamente.
    # Modelo simplificado: El tiempo total se divide entre la cantidad de desarrolladores (sin considerar solapamientos u orden).
    tiempo_total_parallel = tiempo_total_secuencial_horas / num_devs
    
    # Convertimos a días asumiendo horas_por_dia
    dias_requeridos = tiempo_total_parallel / horas_por_dia

    print(f"\n--- Cálculo de días necesarios con {num_devs} desarrollador(es) ---")
    print(f"Tiempo total migración (secuencial, 1 dev): {tiempo_total_secuencial_horas:.2f} horas")
    print(f"Tiempo migración en paralelo con {num_devs} devs: {tiempo_total_parallel:.2f} horas (~ {dias_requeridos:.1f} días)\n")


--- Cálculo de días necesarios con 9 desarrollador(es) ---
Tiempo total migración (secuencial, 1 dev): 14198.86 horas
Tiempo migración en paralelo con 9 devs: 1577.65 horas (~ 197.2 días)



# fuerza bruta calculo

In [93]:
import math

# --------------------------------------------------------------
# 1. Parámetros principales (ajusta según tu realidad)
# --------------------------------------------------------------
# Cantidad total de registros a migrar
cantidad_registros = 2_000_000  # Ejemplo: 2 millones de registros

# Velocidad de migración: registros por hora que un desarrollador puede procesar
velocidad_migracion_registros_hora = 50_000

# Jornada laboral diaria (en horas)
horas_por_dia = 8

# Días límite deseado para terminar la migración (para calcular devs necesarios)
dias_limite = 30

# Número fijo de desarrolladores (para calcular días requeridos).
# Si se deja en None, el script calcula cuántos desarrolladores se necesitan.
desarrolladores_fijos = None  # Ejemplo: None (para calcular devs) o un valor entero

# --------------------------------------------------------------
# 2. Cálculo del tiempo total (si fuera 1 solo desarrollador)
# --------------------------------------------------------------
tiempo_total_horas_1_dev = cantidad_registros / velocidad_migracion_registros_hora

print(f"\nCantidad de Registros: {cantidad_registros}")
print(f"Velocidad de Migración (registros/hora): {velocidad_migracion_registros_hora}")
print(f"Tiempo total en horas con 1 dev (secuencial): {tiempo_total_horas_1_dev:.2f} horas")

# --------------------------------------------------------------
# 3. Cálculo de cuántos desarrolladores se necesitan
#    para terminar en 'dias_limite'
# --------------------------------------------------------------
if desarrolladores_fijos is None:
    # Horas totales disponibles para 1 desarrollador en ese límite
    horas_totales_disponibles_1_dev = dias_limite * horas_por_dia

    # Cuántos desarrolladores se requieren para paralelizar
    # y terminar a tiempo
    devs_requeridos = math.ceil(tiempo_total_horas_1_dev / horas_totales_disponibles_1_dev)

    print(f"\n--- Cálculo de Desarrolladores necesarios en {dias_limite} días ---")
    print(f"Días límite: {dias_limite}")
    print(f"Horas totales disponibles (1 dev): {horas_totales_disponibles_1_dev} horas")
    print(f"Desarrolladores requeridos (aprox): {devs_requeridos}")

# --------------------------------------------------------------
# 4. Cálculo de tiempo requerido si ya tienes
#    un número fijo de desarrolladores
# --------------------------------------------------------------
else:
    num_devs = desarrolladores_fijos

    # Tiempo total en horas dividido por el número de desarrolladores
    tiempo_paralelo_horas = tiempo_total_horas_1_dev / num_devs
    dias_requeridos = tiempo_paralelo_horas / horas_por_dia

    print(f"\n--- Cálculo de Tiempo con {num_devs} desarrollador(es) ---")
    print(f"Tiempo total en paralelo: {tiempo_paralelo_horas:.2f} horas")
    print(f"Días requeridos (aprox): {dias_requeridos:.1f}")



Cantidad de Registros: 2000000
Velocidad de Migración (registros/hora): 100000
Tiempo total en horas con 1 dev (secuencial): 20.00 horas

--- Cálculo de Desarrolladores necesarios en 30 días ---
Días límite: 30
Horas totales disponibles (1 dev): 240 horas
Desarrolladores requeridos (aprox): 1


In [111]:
import math

# ---------------------------------------------
# 1. Parámetros principales (ajustar a la realidad)
# ---------------------------------------------
cantidad_registros = 3_000_000  # Ejemplo: 2 millones de registros totales

velocidad_migracion_registros_hora = 50_000  # Registros/hora que 1 desarrollador puede procesar

horas_por_dia = 5  # Jornada laboral diaria

num_devs = 3       # Cantidad de desarrolladores asignados


# ---------------------------------------------
# 2. Cálculo: Tiempo secuencial con 1 desarrollador
# ---------------------------------------------

tiempo_total_horas_1_dev = cantidad_registros / velocidad_migracion_registros_hora



print(f"\nCantidad de Registros: {cantidad_registros:,}")
print(f"Velocidad de Migración: {velocidad_migracion_registros_hora:,} registros/hora")
print(f"Desarrolladores Asignados: {num_devs}")
print(f"Tiempo total (1 dev, secuencial): {tiempo_total_horas_1_dev:.2f} horas")


# ---------------------------------------------
# 3. Cálculo: Tiempo paralelo con 'num_devs' desarrolladores
# ---------------------------------------------

# Modelo simplificado: se divide la carga total entre todos los desarrolladores

tiempo_total_horas_en_paralelo = tiempo_total_horas_1_dev / num_devs

# Convertir a días

tiempo_total_dias_en_paralelo = tiempo_total_horas_en_paralelo / horas_por_dia


print(f"Tiempo total en paralelo con {num_devs} dev(s): {tiempo_total_horas_en_paralelo:.2f} horas")
print(f"Equivale a ~ {tiempo_total_dias_en_paralelo:.1f} días\n")



Cantidad de Registros: 3,000,000
Velocidad de Migración: 50,000 registros/hora
Desarrolladores Asignados: 3
Tiempo total (1 dev, secuencial): 60.00 horas
Tiempo total en paralelo con 3 dev(s): 20.00 horas
Equivale a ~ 4.0 días



In [122]:
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

In [120]:
import pandas as pd
import numpy as np

In [128]:
import math

# ---------------------------------------------
# 1. Parámetros principales
# ---------------------------------------------
cantidad_registros = 3_000_000

velocidad_migracion_registros_hora = 50_000  # Por desarrollador
horas_por_dia = 5
num_devs = 6
factor_eficiencia = 0.8  # 80% eficiencia por paralelización
buffer_errores = 1.2  # 20% más por reprocesos

# ---------------------------------------------
# 2. Cálculos
# ---------------------------------------------

# Tiempos base (modelo ideal)

tiempo_secuencial = cantidad_registros / velocidad_migracion_registros_hora

tiempo_paralelo_teorico = tiempo_secuencial / num_devs

# Preguntar a Rafel el tema de ciclo y si es fijo
dias_teoricos = tiempo_paralelo_teorico / horas_por_dia

# Tiempos ajustados (modelo realista)
tiempo_paralelo_real = (tiempo_paralelo_teorico / factor_eficiencia) * buffer_errores

dias_reales = tiempo_paralelo_real / horas_por_dia


# ---------------------------------------------
# 3. Resultados (formato claro)
# ---------------------------------------------

print(f"\n[PARÁMETROS INICIALES]")
print(f"Cantidad de Registros: {cantidad_registros:,}")
print(f"Velocidad de Migración: {velocidad_migracion_registros_hora:,} registros/hora por desarrollador")
print(f"Desarrolladores Asignados: {num_devs}")

print(f"\n[ESTIMACIÓN IDEAL]")
print(f"Tiempo total (1 dev, secuencial): {tiempo_secuencial:.2f} horas")
print(f"Tiempo paralelo teórico ({num_devs} devs): {tiempo_paralelo_teorico:.2f} horas")
print(f"Equivale a ~ {dias_teoricos:.1f} días")

print(f"\n[ESTIMACIÓN REALISTA]")
print(f"* Eficiencia en paralelización: {factor_eficiencia*100}%")
print(f"* Buffer por errores/reprocesos: +{(buffer_errores-1)*100:.0f}%")
print(f"\nTiempo total ajustado: {tiempo_paralelo_real:.2f} horas")
print(f"Equivale a ~ {dias_reales:.1f} días\n")


[PARÁMETROS INICIALES]
Cantidad de Registros: 3,000,000
Velocidad de Migración: 50,000 registros/hora por desarrollador
Desarrolladores Asignados: 6

[ESTIMACIÓN IDEAL]
Tiempo total (1 dev, secuencial): 60.00 horas
Tiempo paralelo teórico (6 devs): 10.00 horas
Equivale a ~ 2.0 días

[ESTIMACIÓN REALISTA]
* Eficiencia en paralelización: 80.0%
* Buffer por errores/reprocesos: +20%

Tiempo total ajustado: 15.00 horas
Equivale a ~ 3.0 días



In [131]:
# Bastante tiempo

In [ ]:
import math

# ---------------------------------------------
# 1. Parámetros principales
# ---------------------------------------------
cantidad_registros = 3_000_000

velocidad_migracion_registros_hora = 50_000  # Por desarrollador
horas_por_dia = 5
num_devs = 6
factor_eficiencia = 0.8  # 80% eficiencia por paralelización
buffer_errores = 1.2  # 20% más por reprocesos

# ---------------------------------------------
# 2. Cálculos
# ---------------------------------------------

# Tiempos base (modelo ideal)

tiempo_secuencial = cantidad_registros / velocidad_migracion_registros_hora

tiempo_paralelo_teorico = tiempo_secuencial / num_devs

# Preguntar a Rafel el tema de ciclo y si es fijo
dias_teoricos = tiempo_paralelo_teorico / horas_por_dia

# Tiempos ajustados (modelo realista)
tiempo_paralelo_real = (tiempo_paralelo_teorico / factor_eficiencia) * buffer_errores

dias_reales = tiempo_paralelo_real / horas_por_dia


# ---------------------------------------------
# 3. Resultados (formato claro)
# ---------------------------------------------

print(f"\n[PARÁMETROS INICIALES]")
print(f"Cantidad de Registros: {cantidad_registros:,}")
print(f"Velocidad de Migración: {velocidad_migracion_registros_hora:,} registros/hora por desarrollador")
print(f"Desarrolladores Asignados: {num_devs}")

print(f"\n[ESTIMACIÓN IDEAL]")
print(f"Tiempo total (1 dev, secuencial): {tiempo_secuencial:.2f} horas")
print(f"Tiempo paralelo teórico ({num_devs} devs): {tiempo_paralelo_teorico:.2f} horas")
print(f"Equivale a ~ {dias_teoricos:.1f} días")

print(f"\n[ESTIMACIÓN REALISTA]")
print(f"* Eficiencia en paralelización: {factor_eficiencia*100}%")
print(f"* Buffer por errores/reprocesos: +{(buffer_errores-1)*100:.0f}%")
print(f"\nTiempo total ajustado: {tiempo_paralelo_real:.2f} horas")
print(f"Equivale a ~ {dias_reales:.1f} días\n")